#  HI fluxes

Here we aim to explore the HI fluxes from different catalogs using [edge_pydb](https://github.com/tonywong94/edge_pydb)

1. edge_leda (647 entries, but only 453 have fluxes)
2. edge_hiflux (161 entries; edge2015 and some more)
3. CALIFA_HI_sample_archive.csv (923 entries)
4. hiedge_all.csv (568 entries)


5. FASHI (41741 entries)
6. ALFALFA  (31502 entries) 

Also of use

1. iEDGE (643 entries) 
2. edge_califa (671 entries)
3. amusing (621 entries)

See also (private repo) https://github.com/tonywong94/edge_hispec/


# Tables used below

Below the short table names in the code, the number of entries, and the `key` of the galaxy name for any join/merging operations.

```   
 t0   edge_califa.csv                  671   Name
 t1   edge_leda.csv                    647   Name
 t2   iedge_v1.ecsv                    643   CALIFA_name
 t3   edge_hiflux.csv                  159   Name              only has names, no ra,dec
 t4   edge2025.csv                      43   Name              only has names, no ra,dec
 t5   CALIFA_HI_sample_archive.csv     923   CALIFA_name       562 have flux, 361 have no archived flux
 t6   hiedge_all.csv                   568   Name              only names, from veselina's spectra plots
 t7   amusing_sample_char.csv          621   Amusing_Name      only 20 in overlap with CALIFA
 t8   xCOLDGASS.txt                    532   SDSS Name         xCOLDGASS_PubCat.fits for full catalogue
 a100 a100.code12.table2.190808.csv  31502   -                 probably should use the OC coordinates, not HI
 fast Table2-FASHI*catalog.csv       41741   -

In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from astropy.io import ascii
from astropy.table import QTable, Table, join
from astropy.coordinates import SkyCoord, Distance
from astropy import units as u
try:
    from edge_pydb import EdgeTable
    print("edge_pydb installed")
except:
    print("No edge_pydb installed, no EdgeTable avaiable")
    


In [ ]:
def common(t1, t2, c1, c2, mask=False):
    """ return a selection of t1 where c1 in t1 matches a c2 in t2"""
    mask12 = np.isin(t1[c1], t2[c2])
    n12 = len(t1[mask12])
    print("common",len(t1),len(t2), "->", n12)
    if mask:
        return mask12
    return t1[mask12]

In [ ]:
EdgeTable('list')

In [ ]:
# there's a "bug" in edge_pydb in that it doesn't recognize ecsv yet. I have a PR coming
if True:
    from edge_pydb import util
    util.updatefiles()
    EdgeTable('list')

In [ ]:
# note the comment line became a row
amusing = EdgeTable('amusing_sample_char.csv')
amusing.keys()
amusing

In [ ]:
#leda = pd.read_csv('edge_leda.csv')
#leda = np.loadtxt('edge_leda.csv', skiprows=1, delimiter=',')
#leda = np.genfromtxt('edge_leda.csv', skip_header=1, delimiter=',')
leda = EdgeTable('edge_leda.csv')

In [ ]:
import astropy
astropy.__version__

In [ ]:
leda.info()


In [ ]:
# Name,ledaHIflux
idx = ~leda['ledaHIflux'].mask

# 453 galaxies (out of 647) have a flux
leda_haveflux = leda['Name'][idx]
leda_flux = leda['ledaHIflux'][idx]
#leda_flux


In [ ]:
iedge = EdgeTable("iedge_v1.csv")

In [ ]:
iedge.info()

In [ ]:
edge_hiflux = EdgeTable("edge_hiflux.csv")


In [ ]:
edge_hiflux
# Name, SigInt, SigUnc
edge_hiflux['Name']


In [ ]:
califa = EdgeTable('edge_califa.csv')


In [ ]:
califa

## LEDA vs. EDGE

Take all edge_leda galaxies which have a known flux (no error bars available). Then find the flux in edge_hiflux for those galaxies

In [ ]:
#leda = EdgeTable('edge_leda.csv', cols=['Name', 'ledaHIflux'])
leda = EdgeTable('edge_leda.csv')

In [ ]:
edge_hiflux = EdgeTable("edge_hiflux.csv", cols=['Name', 'SigInt', 'SigUnc'])

In [ ]:
leda.join(edge_hiflux)
len(leda)

In [ ]:
leda.keys()
#print(leda)

In [ ]:
fig, ax = plt.subplots()
ax.plot(leda['ledaHIflux'], leda['SigInt'], 'o')
ax.errorbar(leda['ledaHIflux'], leda['SigInt'], yerr=leda['SigUnc'],ls='none')
fmin = 0
fmax = 20
ax.set_xlim(fmin,fmax)
ax.set_ylim(fmin,fmax)
ax.grid()
ax.set_aspect('equal')
ax.plot([fmin,fmax],[fmin,fmax],'-', color='red')
ax.set_xlabel("LEDA flux")
ax.set_ylabel("EDGE_HIFLUX")

# Cross matching catalogs

The catalogs (t0, t1, ....) are listed at the top of this notebook.

## 0. edge_califa

In [ ]:
t0 = QTable.read('edge_califa.csv', format="ecsv")
t0

## 1. edge_leda

In [ ]:
t1 = QTable.read("edge_leda.csv", format="ecsv")
#t1.keys()[:7]
t1

In [ ]:
t1c = SkyCoord(t1["ledaRA"], t1["ledaDE"])

## 2. iedge


In [ ]:
t2 = QTable.read("iedge_v1.ecsv", format="ecsv")
#t2.keys()[:7]
t2

In [ ]:
t2c = SkyCoord(t2["RA"], t2["DEC"])

Check overlap between t1 and t2 based on their coordinates.   it is surprising that this is not close,
limiting to 10" there are only 478/643.... odd.

In [ ]:
idx, sep, _ = t1c.match_to_catalog_sky(t2c)

(sep < 10 * u.arcsec).sum(), len(t2)

In [ ]:
plt.hist(sep.arcsec, histtype="step", bins=np.logspace(-2, 3, 64))
plt.xlabel("separation [arcsec]")
plt.xscale("log")
plt.yscale("log")
plt.tight_layout()
plt.title('leda and iedge')

## 3. edge_hiflux

In [ ]:
t3 = QTable.read("edge_hiflux.csv", format="ecsv")
t3n = t3["Name"]

In [ ]:
t31 = join(t3,t1)
#t31.keys()
#t31

In [ ]:
flux1 = t31["ledaHIflux"]
flux3 = t31["SigInt"]

plt.plot(flux1,flux3,'o',label="data")
plt.plot([0,50], [0,50], '-', label="1:1")
plt.xlabel("LEDA")
plt.ylabel("GBT 2015+")
#plt.aspect("equal")
plt.legend();
plt.title(f"GBT vs. LEDA: {len(t31)} entries")

## 4. edge2025

Provisional

In [ ]:
t4 = Table.read("edge2025.csv", format="ecsv")

In [ ]:
t41 = join(t4,t1)
len(t41)

In [ ]:
#print(flux4)
#print(flux1)


In [ ]:
flux1 = t41["ledaHIflux"]
flux4 = t41["SigInt"]

idx = np.where(~np.isnan(flux1))
#print(len(cnt[0]))
cnt = len(idx[0])

plt.plot(flux1[idx],flux4[idx],'o',label="data")
plt.plot([0,20], [0,20], '-', label="1:1")
plt.xlabel("LEDA")
plt.ylabel("GBT 2025")
plt.legend();
plt.title(f"GBT25 vs. LEDA: {cnt} entries")

In [ ]:
# panda's style doesn't work for tables
#  t2[t2["CALIFA_name"].isin(t3n)]

# would need to convert to pandas:
# pt2 = t2.to_pandas()


In [ ]:
# Create mask and extract rows
mask = np.isin(t2['CALIFA_name'], t3n)
t2_3 = t2[mask]
len(t2),len(t2_3)

In [ ]:
t1.keys()
t41 = common(t1,t4,"ledaName","Name")
t41

In [ ]:
a=common(t2,t3,"CALIFA_name","Name")
b=common(t3,t2,"Name","CALIFA_name")

In [ ]:
a

In [ ]:
b


## 5. CALIFA_HI_sample_archive

In [ ]:
t5 = QTable.read("CALIFA_HI_sample_archive.csv")
len(t5)

In [ ]:
t5y = t5[t5["Archive_HI?"] == "Y"]
len(t5y)

In [ ]:
t5n = t5[t5["Archive_HI?"] == "N"]
len(t5n)

In [ ]:
t5_c  = SkyCoord(t5['RA']*u.deg, t5['DEC']*u.deg)
t5y_c = SkyCoord(t5y['RA']*u.deg,t5y['DEC']*u.deg)
t5n_c = SkyCoord(t5n['RA']*u.deg,t5n['DEC']*u.deg)

In [ ]:
plt.figure(figsize=(8, 4.2))
ax = plt.subplot(111, projection="aitoff")
ax.grid(True)

coords = t5y_c
ra_rad = coords.ra.wrap_at(180 * u.deg).radian
dec_rad = coords.dec.radian
cnt1 = len(coords)
ax.scatter(ra_rad, dec_rad, color='blue', s=20, label=f"with HI -- {cnt1}")

coords = t5n_c
ra_rad = coords.ra.wrap_at(180 * u.deg).radian
dec_rad = coords.dec.radian
cnt2 = len(coords)
ax.scatter(ra_rad, dec_rad, color='red', s=20, label=f"no HI -- {cnt2}")

ax.legend(loc="lower right")
ax.set_title(f'CALIFA ({cnt1+cnt2})')

## 8. xCOLDGASS



In [ ]:
t8 = QTable.read('xCOLDGASS.txt', format="ascii.tab")

t8_c = SkyCoord(t8['RA']*u.deg,t8['Dec']*u.deg)


In [ ]:
plt.figure(figsize=(8, 4.2))
ax = plt.subplot(111, projection="aitoff")
ax.grid(True)

coords = t8_c
ra_rad = coords.ra.wrap_at(180 * u.deg).radian
dec_rad = coords.dec.radian
cnt = len(coords)
ax.scatter(ra_rad, dec_rad, color='blue', s=20)
ax.legend(loc="lower right")
ax.set_title(f'xCOLDGASS -- {cnt}')

In [ ]:
# comparing xCOLDGAS w/ CALIFA

idx, sep, _ = t5_c.match_to_catalog_sky(t8_c)

sep_max = 60
mask = (sep < sep_max * u.arcsec)
print(t5[mask])
print(len(mask))
#print(mask)

(sep < sep_max * u.arcsec).sum(), len(t8_c)

In [ ]:
t8d = QTable.read('xCOLDGASS_PubCat.fits', format="fits")
t8d


## a100: ALFALFA

In [ ]:
a100 = QTable.read('a100.code12.table2.190808.csv', format="csv")

In [ ]:
a100


In [ ]:
#a100_c = SkyCoord(a100["RAdeg_HI"]*u.deg, a100["DECdeg_HI"]*u.deg)
a100_c = SkyCoord(a100["RAdeg_OC"]*u.deg, a100["DECdeg_OC"]*u.deg)

In [ ]:
plt.figure(figsize=(8, 4.2))
ax = plt.subplot(111, projection="aitoff")
ax.grid(True)

coords = a100_c
ra_rad = coords.ra.wrap_at(180 * u.deg).radian
dec_rad = coords.dec.radian
cnt = len(coords)
ax.scatter(ra_rad, dec_rad, color='blue', s=20)
ax.legend(loc="lower right")
ax.set_title(f'ALFALFA -- {cnt}')

## FAST

FASHI = HI_source catalog from FAST

In [ ]:
# first two rows are header:  one for column name,one for unit
fast = Table.read('Table2-FASHI_extragalactic_HI_source_catalog.csv', format="ascii.csv")
fast.remove_row(0)
fast

In [ ]:
ra = fast["ra"].astype(float)*u.deg
de = fast['dec'].astype(float)*u.deg
fast_c = SkyCoord(ra,de)

In [ ]:
plt.figure(figsize=(8, 4.2))
ax = plt.subplot(111, projection="aitoff")
ax.grid(True)

coords = fast_c
ra_rad = coords.ra.wrap_at(180 * u.deg).radian
dec_rad = coords.dec.radian
cnt = len(coords)
ax.scatter(ra_rad, dec_rad, color='blue', s=20)
ax.legend(loc="lower right")
ax.set_title(f'FASHI -- {cnt}')

In [ ]:
t31 = join(t3,t1)   # 159
t31_c = SkyCoord(t31["ledaRA"], t31["ledaDE"])

In [ ]:
t32 = join(t3,t2, keys_right='CALIFA_name',keys_left='Name'   ) 
t32

In [ ]:
join(t5,t1,keys_left='CALIFA_name',keys_right='Name')


In [ ]:
#t51 = join(t5,t1)

### Comparing CALIFA HI (y/n) with ALFALFA

In [ ]:
idx, sep, _ = t5y_c.match_to_catalog_sky(a100_c)

(sep < 10 * u.arcsec).sum(), len(t5y_c)

In [ ]:
plt.hist(sep.arcsec, histtype="step", bins=np.logspace(-1, 3.0, 64))
plt.xlabel("separation [arcsec]")
plt.xscale("log")
plt.yscale("log")
plt.title(f"Distribution of nearest neighor of CALIFA to Arecibo [arcsec]: count {len(sep)}")
plt.tight_layout()

In [ ]:
idx, sep, _ = t5n_c.match_to_catalog_sky(a100_c)

(sep < 10 * u.arcsec).sum(), len(t5n_c)

## TODO1

15 Galaxies of the Califa_HI=N cases have fluxes from ALFALFA, thus 346 candidates

but we like them to be in iedge (t2), leaving us with 176.

In [ ]:
idx2 = (sep < 10 * u.arcsec)
t5na = t5n[~idx2]

# find the ones common in iedge
mask52 = common(t5na, t2, "CALIFA_name", "CALIFA_name", mask=True)
print(len(mask52))
t5nai = t5na[mask52]


ascii.write(t5nai, 'todo1.txt', include_names=['CALIFA_name'], format='no_header', overwrite=True)

In [ ]:
mask25 = common(t2, t5nai, "CALIFA_name", "CALIFA_name", mask=True)
t2n = t2[mask25]
t2y = t2[~mask25]
print(len(t2n),len(t2y))

In [ ]:

bins=np.linspace(0,8,40)

cnt1 = len(t2n)
cnt2 = len(t2y)

plt.hist(np.log(t2n['APEX_Glob_Fco'].value), bins=bins, alpha=0.4, label=f"no HI: {cnt1}")
plt.hist(np.log(t2y['APEX_Glob_Fco'].value), bins=bins, alpha=0.2, label=f"with HI {cnt2}")
plt.xlim(0,8)
plt.ylim(0,30)
plt.xlabel("log($F_{CO}$)")
plt.title("APEX global Fco")
plt.legend();

There are only 40 with log(Fco) > 3


In [ ]:
t2.keys()

In [ ]:
mask3 = np.log(t2n['APEX_Glob_Fco'].value) > 3
t2n_3 = t2n[mask3]
len(t2n_3)
ascii.write(t2n_3, 'todo2.txt', include_names=['CALIFA_name'], format='no_header', overwrite=True, comment="todo2")

print(mask3)


In [ ]:
see = ['APEX_Glob_Fco', 'APEX_Glob_eps_obs', 'APEX_Glob_RMS']
t2[see]

In [ ]:
fco = t2['APEX_Glob_Fco'].value 
mask = np.where(~np.isnan(fco))
len(mask[0])  # 501
fco = fco[mask]

fco_snr = t2['APEX_Glob_SNR'].value 
fco_snr = fco_snr[mask]

fco_snr = t2['APEX_Glob_eps_obs'].value 
fco_snr = fco_snr[mask]
fco_snr = fco/fco_snr 

In [ ]:
bins=np.linspace(0,8,40)

cnt = len(fco)

plt.hist(np.log(fco), bins=bins)
plt.xlim(0,8)
plt.ylim(0,60)
plt.xlabel("log($F_{CO}$)")
plt.title(f"APEX global Fco ({cnt})")
plt.legend();

In [ ]:
plt.plot(np.log(fco),np.log(fco_snr),'o')
plt.plot([0,5],[0,5])
plt.plot([0,10],[0,5])
plt.axhline(np.log(5), color='black', lw=3)
#plt.xlim(0,8)
#plt.ylim(-1,5)
plt.xlabel("log($F_{CO}$)")
plt.ylabel("log(SNR)")
plt.title(f"APEX global Fco ({cnt})")
plt.legend();

In [ ]:
plt.hist(sep.arcsec, histtype="step", bins=np.logspace(-1, 3.0, 64))
plt.xlabel("separation [arcsec]")
plt.xscale("log")
plt.yscale("log")
plt.title(f"Distribution of nearest neighor of CALIFA to Arecibo [arcsec]: count {len(sep)}")
plt.tight_layout()

In [ ]:
idx, sep, _ = t31_c.match_to_catalog_sky(a100_c)

(sep < 10 * u.arcsec).sum(), len(t31)

In [ ]:
mask = sep < 10 * u.arcsec
mask

In [ ]:
plt.hist(sep.arcsec, histtype="step", bins=np.logspace(-1, 3.0, 64))
plt.xlabel("separation [arcsec]")
plt.xscale("log")
plt.yscale("log")
plt.title(f"Distribution of nearest neighor of CALIFA to Arecibo [arcsec]: count {len(sep)}")
plt.tight_layout()

arguable all less than 10" are spot on. That's 62 out of the 159, not bad.

In [ ]:
mask

There are 23/159 galaxies in CALIFA_HI that seem to overlap with Arecibo, if HI location used. But 62 if the OC location!!

In [ ]:
#a100_near = a100[mask]
a100_near = t31[mask]
print(len(a100_near))
a100_near["sep"] = sep.arcsec[mask]

In [ ]:
#t31.keys()   # Name, SigInt, ledaHIflux

In [ ]:
# add more columns.... I didn't know how to do the join here
a100_near["Name2"] = t31["Name"]
a100_near["SigInt"] = t31["SigInt"]

In [ ]:
a100_near

Cut the 159 down to 62

In [ ]:
a100_near[see]

In [ ]:
see = ["AGCNr","Name","Name2","RAdeg_OC","DECdeg_OC","HIflux", "SigInt", "sep"]
idx2 = a100_near["sep"] < 10
a100_near2 = a100_near[idx2]
a100_near2[see]

In [ ]:
plt.plot(a100_near["HIflux"], a100_near["SigInt"],"o")
plt.plot([0,30],[0,30])
plt.xlabel("ALFALFA  [Jy.km/s]")
plt.ylabel("GBT++   [Jy.km/s]" )
plt.title(f"a100_near:  {len(a100_near['HIflux'])}")

ok, quite a lot of weak arecibo data have good GBT fluxes.  But not the other way around.  Need to see spectra.

In [ ]:
idx = np.where(a100_near["HIflux"] > 30)[0]
a100_near[idx]

## 6. hiedge_all

This is currently just a list of names of all the spectra Veselina obtained from NED

In [ ]:
# names of all the califa galaxies where hi was detected - and where not
t6 = QTable.read('hiedge_all.csv')


In [ ]:
see = ['CALIFA_name', 'APEX_Glob_Fco','CARMA_Glob_Fco','ACA_Glob_Fco']
t2g = t2[see]

In [ ]:
cnt, bins, patches = plt.hist(t2['APEX_Glob_Fco'], bins=32, log=True)
cnt1 = int(cnt.sum())


In [ ]:
cnt, bins, patches = plt.hist(t2['CARMA_Glob_Fco'], bins=32, log=True)
cnt2 = int(cnt.sum())

In [ ]:
cnt, bins, patches = plt.hist(t2['ACA_Glob_Fco'], bins=32, log=True)
cnt3 = int(cnt.sum())

In [ ]:
plt.hist(t2['APEX_Glob_Fco'], bins=32, log=True, label=f'APEX {cnt1}')
plt.hist(t2['CARMA_Glob_Fco'], bins=32, log=True, label=f'CARMA {cnt2}')
plt.hist(t2['ACA_Glob_Fco'], bins=32, log=True,   label=f'ACA {cnt3}')
plt.title("Count of cold gas detections")
plt.legend();


In [ ]:
t32g = join(t3, t2g, keys_left='Name', keys_right='CALIFA_name')

In [ ]:
idx = np.where(~np.isnan(t32g['APEX_Glob_Fco']))
print(idx)

In [ ]:
cnt = len(t32g['SigInt'][idx])

plt.plot(t32g['SigInt'][idx], t32g['APEX_Glob_Fco'][idx],'o')
plt.xlim(0,40)
plt.ylim(0,1500)
plt.xlabel("HI from edge_hiflux [Jy km/s]")
plt.ylabel("APEX_Glob_Fco [K km/s]")
plt.title(f"CO with HI : {cnt}");

How many of our t3 are in t5?

In [ ]:
c53 = common(t5,t3,'CALIFA_name','Name')

In [ ]:
c54 = common(t5,t4,'CALIFA_name','Name')

In [ ]:
c51 = common(t5,t1,'CALIFA_name','ledaName')
c51 = common(t5,t1,'CALIFA_name','Name')

## Comparing FAST and ALFALFA

There's a paper on this too.


In [ ]:
#idx, sep, _ = a100_c.match_to_catalog_sky(fast_c)
%time idx, sep, _ = fast_c.match_to_catalog_sky(a100_c)
(sep < 10 * u.arcsec).sum()

In [ ]:
len(fast_c),len(a100_c)

In [ ]:
plt.hist(sep.arcsec, histtype="step", bins=np.logspace(-1, 4, 64))
plt.xlabel("separation [arcsec]")
plt.xscale("log")
plt.yscale("log")
plt.title(f"Distribution of nearest neighor of ALFALFA vs. FASHI [arcsec]")
plt.tight_layout()